# 02 - Retrieval experiments

Notebook 01 gave you one number. This one gives you a **table**, which is the only way a number
becomes a finding.

The method is the boring, correct one: fix everything, change exactly one thing, re-measure.
Every config under `configs/phase1/` is generated from one shared base in
`scripts/gen_configs.py`, so the only difference between two runs is the knob named in the
filename. That is not fussiness - it is what stops you reporting "bge-small is better" when you
also happened to change the chunk size.

```
                        base config
                             |
        +--------------------+--------------------+
        |                    |                    |
   [1a] EMBEDDER        [1b] CHUNK          [1c] RETRIEVAL
   5 models             5 splits            4 modes
   MiniLM / bge /       whole / 256 /       bm25 / dense /
   e5 / gte / mpnet     512 / 1024 / sent   hybrid / +rerank
        |                    |                    |
        +--------------------+--------------------+
                             |
                        leaderboard()
```

14 runs, 1,000 documents, exact search, **no generation**. Approximate indexes and the scale
ladder are Phase 2; LLM comparison is Phase 3. Both are planned in `context/02-plan.md` but are
not implemented in this repo yet.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib, glob, time
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from fitment_rag.config import RunConfig
from fitment_rag.pipeline import run
from fitment_rag.report import leaderboard

pd.set_option("display.width", 200)

def sweep(pattern, label):
    """Run every config matching a glob. Cached stages make re-runs cheap."""
    paths = sorted(glob.glob(pattern))
    print(f"{label}: {len(paths)} configs\n")
    for i, p in enumerate(paths, 1):
        cfg = RunConfig.load(p)
        print(f"--- [{i}/{len(paths)}] {cfg.name} ---")
        t0 = time.perf_counter()
        try:
            run(cfg, verbose=False)
            print(f"    done in {time.perf_counter()-t0:.1f}s")
        except Exception as exc:
            print(f"    FAILED: {exc}")
    print()

## Before you run anything: write down your prediction

Seriously. Fill this in, run the cell, and do not edit it afterwards. When you write this up,
"I predicted X and got Y" is the sentence that makes a reviewer trust the rest of your numbers.

In [ ]:
PREDICTIONS = {
    "best embedder":        "bge-small",     # <- your guess
    "best chunk size":      "fixed512",
    "dense beats bm25?":    "yes, but by less than 10 points",
    "reranker worth it?":   "yes on mrr, no on recall@5",
}
for k, v in PREDICTIONS.items():
    print(f"{k:22} {v}")

## Phase 1a - Embedding models

Five models, all small enough for a 15W CPU. They differ in parameters, output dimension, and
training recipe.

| tag | model | params | dim | note |
|---|---|---|---|---|
| minilm-l6 | all-MiniLM-L6-v2 | 22M | 384 | the fast default |
| bge-small | BAAI/bge-small-en-v1.5 | 33M | 384 | needs a query prefix |
| e5-small | intfloat/e5-small-v2 | 33M | 384 | needs `query:`/`passage:` prefixes |
| gte-small | thenlper/gte-small | 33M | 384 | no prefix |
| mpnet-base | all-mpnet-base-v2 | 110M | 768 | the slow ceiling |

**The prefixes are not cosmetic.** bge and e5 were trained with asymmetric instructions, and
omitting them silently costs several points of recall - one of the most common ways a public
embedding comparison gets quietly rigged. They are set in the configs; check
`cfg.embedding.query_prefix` if a model underperforms.

First run downloads each model (30-500MB) and embeds ~2,400 chunks. Budget 10-20 minutes total.

In [ ]:
sweep("../configs/phase1/emb/*.yaml", "Phase 1a - embedders")
leaderboard(sort="recall@5")

In [ ]:
df = leaderboard()
emb = df[df["run"].str.startswith("emb-")].sort_values("recall@5")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.barh(emb["embedder"], emb["recall@5"], color="#4C78A8")
ax1.set_xlabel("recall@5"); ax1.set_title("Quality")

ax2.scatter(emb["embed_s"], emb["recall@5"], s=90, color="#E45756")
for _, r in emb.iterrows():
    ax2.annotate(r["embedder"], (r["embed_s"], r["recall@5"]),
                 fontsize=8, xytext=(4, 4), textcoords="offset points")
ax2.set_xlabel("seconds to embed the corpus"); ax2.set_ylabel("recall@5")
ax2.set_title("Quality vs cost -- the plot that decides Phase 2")
plt.tight_layout(); plt.show()

**How to read the right-hand plot.** The winner of Phase 1a is not the highest point, it is
the best point you can afford to run 200 times. If mpnet-base buys two points of recall for 5x
the embedding time, it loses - because Phase 2 scales the corpus 200x and that cost scales with it.

Whichever model you pick, say *why* in the writeup. "Best accuracy per second at this corpus
size" is a defensible criterion. "Highest number" is not.

## Phase 1b - Chunking

Now hold the embedder fixed and vary the split. Watch two columns together: `recall@5` and
`chunks`. More chunks means more vectors, more RAM, and slower search, so a chunking strategy
that wins by 1 point while tripling the index is not obviously winning.

In [ ]:
sweep("../configs/phase1/chunk/*.yaml", "Phase 1b - chunking")

df = leaderboard()
df[df["run"].str.startswith("chunk-")][
    ["run", "chunking", "chunks", "recall@5", "mrr", "ndcg@5", "ms/query"]
].sort_values("recall@5", ascending=False)

## Phase 1c - Retrieval mode

The honesty check. **BM25 is a 1994 keyword algorithm with no neural network at all.** If your
embedding stack cannot beat it, that is the single most useful thing this benchmark can tell you -
and in a domain full of part numbers like `PF-2135`, exact lexical matching is genuinely strong.

`hybrid-rerank` adds a cross-encoder over the top-50 candidates. It usually helps MRR more than
recall (it reorders what was already found, it cannot find anything new) and it is the slowest
option by a wide margin - watch `ms/query`.

In [ ]:
sweep("../configs/phase1/retrieval/*.yaml", "Phase 1c - retrieval mode")

df = leaderboard()
ret = df[df["run"].str.startswith("ret-")][
    ["run", "retrieval", "recall@5", "mrr", "ndcg@5", "ms/query"]
].sort_values("recall@5", ascending=False)
display(ret)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(ret))
ax.bar([i - 0.2 for i in x], ret["recall@5"], 0.4, label="recall@5", color="#4C78A8")
ax.bar([i + 0.2 for i in x], ret["mrr"], 0.4, label="mrr", color="#F58518")
ax.set_xticks(list(x)); ax.set_xticklabels(ret["retrieval"], rotation=15)
ax.legend(); ax.set_title("Does dense retrieval actually earn its keep?")
plt.tight_layout(); plt.show()

## The full leaderboard

In [ ]:
final = leaderboard(sort="recall@5")
display(final)

final.to_csv("../results/leaderboard.csv", index=False)
print("wrote results/leaderboard.csv")

---

## Writing this up

Four sentences your README needs, each backed by a row in that table:

1. **The baseline.** What does BM25 get? Every neural number is meaningless without it.
2. **The biggest lever.** Which single knob moved recall@5 most? Name the delta.
3. **The surprise.** What contradicted your `PREDICTIONS` cell? This is the most valuable
   paragraph you will write - it is evidence you ran the experiment rather than decorated a
   conclusion.
4. **The cost.** What did the best config cost in embed time, index size, and ms/query? A
   quality number without its price tag is half a result.

State the limits plainly: one corpus, template-generated questions, a single seed, one machine,
1,000 documents, exact search only. Naming your limitations makes the numbers you *do* report
more credible, not less.

**What Phase 1 hands forward:** the winning embedder, chunk strategy, and retrieval mode -
frozen, so a later phase can vary only the index and the corpus size. Phases 2 and 3 are planned
in `context/02-plan.md` and not implemented in this repo.